# 08: Multi-Protein Training and Evaluation

This experiment trains on several proteins and evaluates proteins excluded from optimization. It asks whether the compact model generalizes its teacher-distillation objective beyond the training set.

Agreement with teacher structures is not the same as validation against experimental structures, and a small held-out set is not enough to claim AlphaFold-level folding performance.

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "../src")

from af2_from_scratch import AF2Config, AlphaFold2FromScratch
from af2_from_scratch.dataset import ProteinDataset
from af2_from_scratch.feature_extraction import sample_batch
from af2_from_scratch.geometry import kabsch_align, kabsch_rmsd

plt.rcParams["figure.figsize"] = (10, 4)
device = "cuda" if torch.cuda.is_available() else "cpu"
split_path = Path("../configs/splits/val.txt")
validation_names = set(split_path.read_text().split()) if split_path.exists() else set()

## 1. Train with an explicit split

Populate `data/` with `scripts/fetch_data.py`, then train from the repository root:

```bash
python scripts/train_multi.py --tag multi 2>&1 | tee logs/train_multi.log
```

The tracked files under `configs/splits/` define included, excluded, and validation proteins. `scripts/train_multi.py` samples only training names for gradient updates and periodically evaluates all loaded proteins.

The cells below skip cleanly when the log, checkpoint, or local data are absent.

## 2. Inspect training and validation curves

Training RMSD describes the protein sampled at each logged step. Validation RMSD is measured periodically with fixed MSA sampling and no masking. Both are agreement with teacher C-alpha coordinates after rigid alignment.

In [ ]:
def parse_log(path="../logs/train_multi.log"):
    path = Path(path)
    if not path.exists():
        print(f"no log at {path}; launch train_multi.py first")
        return {}, {}

    train, validation = {}, {}
    step_pattern = re.compile(r"step\s+(\d+) \| (\S+)\s+\|.*RMSD\s+([\d.]+)")
    score_pattern = re.compile(r"([^\s:'\[\],]+):([\d.]+)")
    current_step = 0
    for line in path.read_text().splitlines():
        match = step_pattern.search(line)
        if match:
            current_step = int(match.group(1))
            train.setdefault(match.group(2), []).append(
                (current_step, float(match.group(3)))
            )
        if ">> VAL" in line:
            for name, rmsd in score_pattern.findall(line):
                validation.setdefault(name, []).append((current_step, float(rmsd)))
    return train, validation


train_history, validation_history = parse_log()
if train_history or validation_history:
    figure, axes = plt.subplots(1, 2, figsize=(15, 4))
    for name, points in train_history.items():
        steps, rmsds = zip(*points)
        axes[0].plot(steps, rmsds, alpha=0.6, label=name)
    for name, points in validation_history.items():
        steps, rmsds = zip(*points)
        axes[0].plot(steps, rmsds, lw=3, label=f"validation: {name}")
    axes[0].set_yscale("log")
    axes[0].set_title("C-alpha RMSD to teacher (Å)")
    axes[0].set_xlabel("step")
    axes[0].legend(fontsize=8)
    if validation_history:
        latest = {name: points[-1][1] for name, points in validation_history.items()}
        axes[1].bar(latest.keys(), latest.values(), color="tomato")
        axes[1].set_title("latest validation RMSD")
        axes[1].set_ylabel("RMSD (Å)")
    plt.tight_layout()
    plt.show()

## 3. Load a checkpoint and evaluate every protein

Change `checkpoint_path` if training used a different `--tag`. Evaluation uses three recycles, meaning four Evoformer passes in total.

In [ ]:
checkpoint_path = Path("../checkpoints/multi.pt")
model = None
dataset = None
targets = None
full_scores = {}

if not checkpoint_path.exists():
    print(f"no checkpoint at {checkpoint_path}; train or update checkpoint_path")
elif not Path("../data").exists():
    print("no data directory; run scripts/fetch_data.py first")
else:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = AF2Config(**checkpoint["cfg"])
    model = AlphaFold2FromScratch(cfg).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    dataset = ProteinDataset("../data")
    targets = {
        name: {key: value.to(device) for key, value in dataset.targets[name].items()}
        for name in dataset.names
    }
    print(f"loaded {checkpoint_path.name} from step {checkpoint.get('step', '?')}")


@torch.no_grad()
def evaluate(query_only=False):
    scores = {}
    for name in dataset.names:
        features = dataset.features[name]
        if query_only:
            features = dict(features)
            features["profile"] = features["msa_aatype"][0]
            n_clu, n_ext = 1, 0
        else:
            n_clu, n_ext = cfg.n_clu, cfg.n_ext
        batch = {
            key: value.to(device)
            for key, value in sample_batch(
                features,
                n_clu,
                n_ext,
                mask_p=0.0,
                seed=42,
            ).items()
        }
        prediction = model(batch, recycles=3)["ca"]
        scores[name] = kabsch_rmsd(prediction, targets[name]["CA"]).item()
    return scores


if model is not None and dataset.names:
    full_scores = evaluate()
    names = dataset.names
    colors = ["tomato" if name in validation_names else "steelblue" for name in names]
    plt.bar(names, [full_scores[name] for name in names], color=colors)
    plt.ylabel("C-alpha RMSD to teacher (Å)")
    plt.title("full MSA; red proteins are held out")
    plt.xticks(rotation=45, ha="right")
    plt.show()
    print({name: round(score, 2) for name, score in full_scores.items()})

## 4. Measure dependence on homologous sequences

A query-only ablation removes homolog rows from both the main and extra MSA and replaces the full-MSA profile with the query sequence itself. The model still receives the query sequence and residue indices.

The comparison measures how much predictions depend on homolog information. Poor query-only performance does not by itself prove correct evolutionary reasoning, and good query-only performance does not by itself prove memorization; sequence-based generalization is also possible.

In [ ]:
if not full_scores:
    print("checkpoint-dependent ablation skipped")
else:
    query_only_scores = evaluate(query_only=True)
    positions = range(len(names))
    width = 0.35
    plt.bar(
        [position - width / 2 for position in positions],
        [full_scores[name] for name in names],
        width,
        label="full MSA",
        color="steelblue",
    )
    plt.bar(
        [position + width / 2 for position in positions],
        [query_only_scores[name] for name in names],
        width,
        label="query only",
        color="orange",
    )
    plt.xticks(list(positions), names, rotation=45, ha="right")
    plt.ylabel("C-alpha RMSD to teacher (Å)")
    for index, name in enumerate(names):
        if name in validation_names:
            plt.axvspan(index - 0.5, index + 0.5, color="tomato", alpha=0.15)
    plt.legend()
    plt.title("homolog-information ablation; shading marks held-out proteins")
    plt.show()

## 5. Inspect held-out traces

The plots below align each prediction to its teacher before visualization. Their titles report RMSD, not mean coordinate error.

In [ ]:
@torch.no_grad()
def fold(name):
    batch = {
        key: value.to(device)
        for key, value in sample_batch(
            dataset.features[name],
            cfg.n_clu,
            cfg.n_ext,
            mask_p=0.0,
            seed=42,
        ).items()
    }
    prediction = model(batch, recycles=3)["ca"]
    teacher = targets[name]["CA"]
    return (
        kabsch_align(prediction, teacher).cpu(),
        teacher.cpu(),
        kabsch_rmsd(prediction, teacher).item(),
    )


held_out_names = (
    [name for name in dataset.names if name in validation_names]
    if dataset is not None
    else []
)
if not held_out_names:
    print("no loaded held-out proteins to plot")
else:
    figure = plt.figure(figsize=(6 * len(held_out_names), 5))
    for index, name in enumerate(held_out_names):
        predicted_ca, teacher_ca, rmsd = fold(name)
        axis = figure.add_subplot(1, len(held_out_names), index + 1, projection="3d")
        axis.plot(*teacher_ca.numpy().T, "o-", ms=4, lw=1.2, label="teacher")
        axis.plot(*predicted_ca.numpy().T, "s-", ms=3, lw=1.2, label="student")
        axis.set_title(f"{name}; held out; RMSD {rmsd:.2f} Å")
        axis.legend()
    plt.tight_layout()
    plt.show()

## 6. Explore MSA depth and error

MSA depth may affect prediction quality because deeper alignments contain more evolutionary observations. This plot is descriptive rather than causal: protein length, family, teacher quality, redundancy, and training-set similarity are confounders.

In [ ]:
if not full_scores:
    print("checkpoint-dependent MSA-depth plot skipped")
else:
    depths = {name: dataset.features[name]["msa_aatype"].shape[0] for name in names}
    lengths = {name: dataset.features[name]["msa_aatype"].shape[1] for name in names}
    plt.scatter(
        [depths[name] for name in names],
        [full_scores[name] for name in names],
        s=[lengths[name] for name in names],
        c=["tomato" if name in validation_names else "steelblue" for name in names],
    )
    for name in names:
        plt.annotate(
            name,
            (depths[name], full_scores[name]),
            fontsize=8,
            xytext=(5, 3),
            textcoords="offset points",
        )
    plt.xscale("log")
    plt.xlabel("MSA depth (log scale)")
    plt.ylabel("C-alpha RMSD to teacher (Å)")
    plt.title("marker size shows protein length; red proteins are held out")
    plt.show()

## 7. Interpretation checklist

Use several observations together:

1. Compare training and held-out teacher agreement across multiple checkpoints.
2. Compare full-MSA and query-only performance on both groups.
3. Repeat training with different random seeds before interpreting small differences.
4. Check whether conclusions hold across more proteins and sequence families.
5. Treat pLDDT calibration separately from coordinate accuracy.

Useful next experiments include increasing the number of training proteins, changing model capacity while holding data fixed, and varying MSA depth without changing the evaluation split.